# UnifyWeaver による家系図チュートリアル

この対話型ノートブックでは、UnifyWeaver を使用して Prolog の述語を Bash スクリプトへとコンパイルする方法を実演します。

## 前提条件

- SWI-Prolog がインストールされていること
- UnifyWeaver ライブラリが利用可能であること
- Prolog Jupyter カーネルがインストールされていること（`pip install prolog-jupyter-kernel`）

## 学習目標

このノートブックを完了すると、以下のことができるようになります:
1. Prolog の事実と規則を定義する
2. UnifyWeaver を使用して述語を Bash にコンパイルする
3. 生成された Bash スクリプトをテストする
4. 推移閉包のコンパイルの仕組みを理解する

## ステップ 1: UnifyWeaver 環境の初期化

まず、UnifyWeaver モジュールをロードする必要があります。education ディレクトリの `init.pl` ファイルを使用します。

In [ ]:
% Load the initialization file
['../init'].

## ステップ 2: 家族関係の定義

聖書の家系図から、いくつかの親子関係（parent-child）を定義してみましょう。

In [ ]:
% Define parent facts
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## ステップ 3: parent クエリのテスト

コンパイルを行う前に、いくつかの Prolog クエリを実行してデータが正しいことを確認します。

In [ ]:
% Query: Who are Abraham's children?
parent(abraham, Child).

In [ ]:
% Query: Who are Jacob's children?
parent(jacob, Child).

## ステップ 4: ancestor（祖先）関係の定義

次に、推移閉包である `ancestor` 関係を定義します。

In [ ]:
% Define ancestor as transitive closure of parent
:- dynamic ancestor/2.

% Base case: parent is an ancestor
ancestor(X, Y) :- parent(X, Y).

% Recursive case: if X is parent of Y and Y is ancestor of Z, then X is ancestor of Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## ステップ 5: ancestor クエリのテスト

ancestor 述語が正しく動作することを確認します。

In [ ]:
% Query: Is Abraham an ancestor of Jacob?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Query: Who are all of Abraham's descendants?
ancestor(abraham, Descendant).

## ステップ 6: parent を Bash にコンパイル

いよいよ本題です — `parent/2` 事実を Bash スクリプトへとコンパイルしてみましょう！

In [ ]:
% Load the stream compiler
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compile parent facts to bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## ステップ 7: parent スクリプトの保存

生成された Bash コードをファイルに保存します。

In [ ]:
% Save to file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## ステップ 8: ancestor を Bash にコンパイル

次に、再帰を使用している `ancestor/2` 述語をコンパイルします。

In [ ]:
% Load the recursive compiler
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compile ancestor to bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## ステップ 9: ancestor スクリプトの保存

ancestor スクリプトをファイルに保存します。

In [ ]:
% Save to file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## ステップ 10: 生成されたスクリプトのテスト

生成された Bash スクリプトをテストしてみましょう！`%%bash` マジックコマンドを使用して Bash コマンドを実行します。

In [ ]:
%%bash
# Source the parent script
source ../output/parent.sh

# Test: Who are Abraham's children?
echo "Abraham's children:"
parent abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Who are Abraham's descendants?
echo "Abraham's descendants:"
ancestor abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Is Abraham an ancestor of Judah?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Yes, Abraham is an ancestor of Judah"
else
    echo "✗ No"
fi

## ステップ 11: コンパイル戦略の理解

UnifyWeaver が行った処理を分析してみましょう:

1. **parent のコンパイル**: `stream_compiler` を使用して、すべての親子ペアを出力するシンプルなストリーミング関数を作成しました

2. **ancestor のコンパイル**: 推移閉包パターンを検出し、幅優先探索（BFS）による最適化を適用して、到達可能なすべての祖先を効率的に計算しました

コンパイル戦略を確認してみましょう:

In [ ]:
% Check if ancestor is classified as recursive
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## まとめ

このノートブックで学んだこと:

✅ Prolog の事実と規則を定義する方法

✅ 事実に対して UnifyWeaver の `stream_compiler` を使用する方法

✅ 再帰的述語に対して UnifyWeaver の `recursive_compiler` を使用する方法

✅ 生成された Bash スクリプトをテストする方法

✅ UnifyWeaver が推移閉包を自動検出し、BFS 最適化を適用すること

## 次のステップ

以下の演習に挑戦してみましょう:

1. 家系図にさらに家族メンバーを追加する
2. `grandparent/2`（祖父母）述語を定義してコンパイルする
3. `sibling/2`（兄弟姉妹: 親が同じ2人）述語を作成する
4. 生成された Bash コードを調べて、BFS アルゴリズムの動作を理解する

高度な再帰パターンについて学ぶために、**ノートブック 2: 再帰パターンの比較** へ進みましょう！